# EEG Denoising

1. Notch filter with (50Hz)60Hz and bandpass filter with 0.5(-1)-30Hz(-50Hz) → this removes powerline noise and generally removes all frequencies that are unwanted in our data (This is done before the segmentation because of the 5s break in between)
2. Remove Artificats with ICA
3. Remove extremely high amplitudes with Amplitude based epoch rejection
4. Normalize data by applying a z-transform
5. Visual inspection


## 0. Load the data

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
PATH_TO_DATA = "../../data/eeg_data_v3_denoised"
PATH_TO_DATA_CLEANED = "../../data/eeg_data_v5_denoised"
PATH_TO_DATA_CLEANED_2 = "../../data/eeg_data_v6_denoised"

In [ ]:
df = pd.read_pickle(f"{PATH_TO_DATA}/pickle/001.pkl")
df_cleaned = pd.read_pickle(f"{PATH_TO_DATA_CLEANED}/pickle/001.pkl")
df_cleaned_2 = pd.read_pickle(f"{PATH_TO_DATA_CLEANED_2}/pickle/001.pkl")
df

In [ ]:
buf = []
for index, row in df.iterrows():
  for i in range(8):
    key = f"EEG_Ch{i+1}"
    buf.append(row[key])
  break

In [ ]:
np.shape(buf)

In [ ]:
from code.feature_extraction.feature_extraction import feature_extraction

ret = feature_extraction(buf)
ret

In [ ]:
df_cleaned

In [ ]:
df_cleaned_2

In [ ]:
# Check whether the shape seems right
print(len(df.loc["Song0", "EEG_Ch1"]))
print(len(df_cleaned.loc["Song0", "EEG_Ch1"]))
print(len(df_cleaned_2.loc["Song0", "EEG_Ch1"]))

In [ ]:
sample_rate = 250
for i in range (8): # 8 channels
  song_one = df.loc["Song0", f"EEG_Ch{i+1}"]
  song_one_cleaned = df_cleaned.loc["Song0", f"EEG_Ch{i+1}"]
  song_one_cleaned_2 = df_cleaned_2.loc["Song0", f"EEG_Ch{i+1}"]
  times = np.arange(len(song_one)) / sample_rate
  fig, (ax2, ax3) = plt.subplots(1, 2, figsize=(30, 3))
  # ax1.plot(times, song_one)
  # ax1.set_title(f"Original Signal EEG_Ch{i+1}")
  # ax1.margins(0, .1)
  # ax1.grid(alpha=.5, ls='--')

  ax2.plot(times, song_one_cleaned)
  ax2.set_title(f"Cleaned EEG_Ch{i+1}")
  ax2.margins(0, .1)
  ax2.grid(alpha=.5, ls='--')

  ax3.plot(times, song_one_cleaned_2)
  ax3.set_title(f"Cleaned EEG_Ch{i+1}")
  ax3.margins(0, .1)
  ax3.grid(alpha=.5, ls='--')
  plt.show()

In [ ]:
# For frequency filtering, all signal data has to be concatanated since we have to look at the whole session
# we therefore need to concatenate all arrays of each channel and store in a list, where each segment starts/ends

def get_concatentated_channel_data(channel_name):
  column = df[channel_name] # get the channel
  ret = []
  offsets = []
  for i in range(len(column)):
    arr = column.iloc[i]
    ret.extend(arr)
    offsets.append(len(arr))
  return ret, offsets

def get_concatentated_channel_data_cleaned(channel_name):
  column = df_cleaned[channel_name] # get the channel
  ret = []
  offsets = []
  for i in range(len(column)):
    arr = column.iloc[i]
    ret.extend(arr)
    offsets.append(len(arr))
  return ret, offsets


In [ ]:
channel_data_list = []
cleaned_channel_data_list = []
for i in range(1, 9, 1):
  concatenated_channel_data, offsets = get_concatentated_channel_data(f"EEG_Ch{i}")
  channel_data_list.append(concatenated_channel_data)
  concatenated_channel_data_cleaned, offsets_cleaned = get_concatentated_channel_data_cleaned(f"EEG_Ch{i}")
  cleaned_channel_data_list.append(concatenated_channel_data_cleaned)

In [ ]:
print(np.shape(channel_data_list))
print(np.shape(cleaned_channel_data_list))

In [ ]:
channel_data_list_np = np.array(channel_data_list)
cleaned_channel_data_list_np = np.array(cleaned_channel_data_list)

In [ ]:
# visualize channels to see whether it is denoised
sample_rate = 250
for i in range (channel_data_list_np.shape[0]):
  times = np.arange(len(channel_data_list_np[i]))/sample_rate
  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(30, 3))
  ax1.plot(times, channel_data_list_np[i])
  ax1.set_title(f"Original Signal EEG_Ch{i+1}")
  ax1.margins(0, .1)
  ax1.grid(alpha=.5, ls='--')

  ax2.plot(times, cleaned_channel_data_list_np[i])
  ax2.set_title(f"Cleaned EEG_Ch{i+1}")
  ax2.margins(0, .1)
  ax2.grid(alpha=.5, ls='--')
  plt.show()

In [ ]:

for i in range(channel_data_list_np.shape[0]):
    times = np.arange(len(channel_data_list_np[i])) / sample_rate

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(30, 6), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

    original = channel_data_list_np[i]
    cleaned = cleaned_channel_data_list_np[i]
    diff = original - cleaned

    # Plot Original and Cleaned signals
    ax1.plot(times, original, label='Original', alpha=0.6)
    ax1.plot(times, cleaned, label='Cleaned', alpha=0.6)
    ax1.set_title(f"EEG Channel {i+1}: Original vs. Cleaned")
    ax1.margins(0, .1)
    ax1.grid(alpha=.5, ls='--')
    ax1.legend()

    # Plot Difference
    ax2.plot(times, diff, label='Difference (Original - Cleaned)', linestyle='--', color='black', alpha=0.7)
    ax2.set_title("Difference")
    ax2.margins(0, .1)
    ax2.grid(alpha=.5, ls='--')
    ax2.legend()

    plt.tight_layout()
    plt.show()